<a href="https://colab.research.google.com/github/tonyOehme/hcc_practical_course_semeval2027/blob/main/Kopie_von_Version_2_subtask_1a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
import glob, re

DATA_DIR = "/content"   # folder where you uploaded the files

def find_file(pattern):
    files = glob.glob(f"{DATA_DIR}/**/{pattern}", recursive=True)
    if not files:
        raise FileNotFoundError(f"No file matching '{pattern}' found in {DATA_DIR}")
    return files[0]

# ---------- load the files ----------
usages      = pd.read_json(find_file("*usages.jsonl"), lines=True)
gold        = pd.read_json(find_file("*subtask1.jsonl"), lines=True)
definitions = pd.read_json(find_file("*definitions.jsonl"), lines=True)
counts      = pd.read_csv(find_file("*usages_count.tsv"), sep="\t")

print("Usages:", len(usages), "| Words:", usages["word"].nunique())

# ---------- check / repair target positions ----------
def fix_span(row):
    s, e = int(row["start"]), int(row["end"])
    if row["text"][s:e].strip().lower()[:2] == row["word"].lower()[:2]:
        return pd.Series([s, e])
    m = (re.search(r"\b" + re.escape(row["word"]) + r"\w*", row["text"], flags=re.IGNORECASE) or
         re.search(r"\b" + re.escape(row["word"][:-1]) + r"\w*", row["text"], flags=re.IGNORECASE))
    return pd.Series([m.start(), m.end()]) if m else pd.Series([s, e])

usages[["start", "end"]] = usages.apply(fix_span, axis=1)
usages["target_found"] = [t[s:e] for t, s, e in zip(usages["text"], usages["start"], usages["end"])]

# ---------- attach gold labels ----------
df = usages.merge(gold[["sentence_id", "label"]], on="sentence_id", how="left")
print("Usages without gold label:", df["label"].isna().sum())
df[["word", "target_found", "period_label", "label", "text"]].head(10)

Usages: 1983 | Words: 10
Usages without gold label: 0


,word,target_found,period_label,label,text
0,kvinna,kvinnor,1880-1883,[3],nesliga lif medelst sjelfmord ; och ett stort ...
1,kvinna,kvinna,1912-1915,[3],"— Jag sov ; jag väcktes av en kylande känsla ,..."
2,kvinna,kvinnans,1880-1883,[3],"orden , så mycken fasthet öfver den gamla kvin..."
3,kvinna,kvinnor,1912-1915,[3],Dessa klubbar ha anordnat sällskapliga samkväm...
4,kvinna,kvinnan,2006-2007,[3],Den avvisningshotade eritranska kvinnan Abiba ...
5,kvinna,kvinnornas,1912-1915,[4],talade på Norrlands nation i söndags på ett mö...
6,kvinna,Kvinnan,2022-2023,[3],Kvinnan medger att hon spelat musik men menar ...
7,kvinna,kvinnan,2006-2007,[3],Läkaren bokade aldrig någon tid och när kvinna...
8,kvinna,kvinna,2006-2007,[3],En 21-årig kvinna avled natten till tisdagen a...
9,kvinna,kvinnor,2022-2023,"[3, 4]",Tre av fyra kvinnor som blivit intervjuade i e...


In [3]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)
model = SentenceTransformer("sachinn1/xl-durel", device=device)

def tag_target(text, start, end):
    return text[:start] + "<t>" + text[start:end] + " </t>" + text[end:]

df["tagged"] = [tag_target(t, int(s), int(e)) for t, s, e in zip(df["text"], df["start"], df["end"])]
print("Example:", df["tagged"].iloc[0])

emb = model.encode(df["tagged"].tolist(), batch_size=32,
                   show_progress_bar=True, normalize_embeddings=True)
df["embedding"] = list(emb)
np.save("xldurel_embeddings.npy", emb)

Using: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.05k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/703 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Example: nesliga lif medelst sjelfmord ; och ett stort antal olyckliga <t>kvinnor </t> , sorn sökt fly undan eländet , ha fallit i „mordeHglarnes4 händer , och deras ben vittra öfver allt i Utahs solbrända dalar .


Batches:   0%|          | 0/62 [00:00<?, ?it/s]

In [4]:
from sklearn.cluster import AgglomerativeClustering

def cluster_word(X, threshold):
    if len(X) < 2:
        return np.zeros(len(X), dtype=int)
    return AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",
                                   distance_threshold=threshold).fit_predict(X)

def bcubed(gold_labels, pred):
    """Extended B-Cubed: works when a usage has several gold senses, e.g. [3, 4]."""
    gold_sets = [set(g) for g in gold_labels]
    pred = np.asarray(pred)
    P, R = [], []
    for i in range(len(pred)):
        same_cluster = pred == pred[i]
        shared = np.array([len(gold_sets[i] & g) for g in gold_sets])
        P.append(np.mean(np.minimum(1, shared[same_cluster])))
        has_shared = shared > 0
        R.append(np.mean(np.where(same_cluster[has_shared], 1 / shared[has_shared], 0)))
    P, R = np.mean(P), np.mean(R)
    return 2 * P * R / (P + R), P, R

def evaluate(threshold):
    rows = []
    for word, g in df.groupby("word"):
        pred = cluster_word(np.vstack(g["embedding"].values), threshold)
        f, p, r = bcubed(g["label"].tolist(), pred)
        base = bcubed(g["label"].tolist(), [0] * len(g))[0]   # baseline: everything in 1 cluster
        rows.append({"word": word, "usages": len(g), "pred_clusters": len(set(pred)),
                     "gold_senses": g["label"].explode().nunique(),
                     "F1": f, "P": p, "R": r, "baseline_1cluster": base})
    return pd.DataFrame(rows)

# try many thresholds (fast: embeddings are already computed)
results = []
for t in np.arange(0.1, 1.01, 0.05):
    r = evaluate(t)
    results.append({"threshold": round(t, 2), "mean_F1": r["F1"].mean(),
                    "mean_clusters": r["pred_clusters"].mean()})
sweep = pd.DataFrame(results)
print(sweep.round(3).to_string(index=False))

BEST = sweep.loc[sweep["mean_F1"].idxmax(), "threshold"]
print("\nBest threshold:", BEST)

 threshold  mean_F1  mean_clusters
      0.10    0.371           45.4
      0.15    0.598           26.1
      0.20    0.672           17.0
      0.25    0.710           11.9
      0.30    0.752            8.9
      0.35    0.799            6.4
      0.40    0.805            4.9
      0.45    0.812            3.8
      0.50    0.805            3.1
      0.55    0.799            2.6
      0.60    0.798            2.3
      0.65    0.796            1.9
      0.70    0.794            1.6
      0.75    0.784            1.2
      0.80    0.784            1.2
      0.85    0.772            1.1
      0.90    0.756            1.0
      0.95    0.756            1.0
      1.00    0.756            1.0

Best threshold: 0.45


In [5]:
res = evaluate(BEST)
print(res.round(3).to_string(index=False))
print("\nMean B-Cubed F1 (model):     ", round(res["F1"].mean(), 3))
print("Mean B-Cubed F1 (1 cluster): ", round(res["baseline_1cluster"].mean(), 3))

df["cluster"] = -1
for word, g in df.groupby("word"):
    df.loc[g.index, "cluster"] = cluster_word(np.vstack(g["embedding"].values), BEST)

df.drop(columns=["embedding", "tagged"], errors="ignore").to_csv("xldurel_subtask1a_clusters.csv", index=False)
print("\nSaved: xldurel_subtask1a_clusters.csv")

     word  usages  pred_clusters  gold_senses    F1     P     R  baseline_1cluster
    dumpa     145              6            6 0.884 0.915 0.855              0.849
exportera     149              1            3 0.946 0.898 1.000              0.946
      fru     274              5            5 0.887 0.843 0.937              0.896
   fröken     174              1            4 0.819 0.694 0.998              0.819
   förort     150              1            3 0.804 0.723 0.906              0.804
    herre     283              4            8 0.868 0.793 0.959              0.815
   kvinna     290              1            5 0.787 0.648 1.000              0.787
    motiv     291              3            5 0.937 0.910 0.967              0.782
     skär      98              8            6 0.544 0.663 0.462              0.545
     suga     129              8           12 0.646 0.558 0.765              0.320

Mean B-Cubed F1 (model):      0.812
Mean B-Cubed F1 (1 cluster):  0.756

Saved: xldure

In [6]:
import json

with open("subtask1a_predictions.jsonl", "w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        f.write(json.dumps({"word": r["word"],
                            "sentence_id": r["sentence_id"],
                            "label": [str(int(r["cluster"]))]}, ensure_ascii=False) + "\n")

with open("subtask1a_predictions.jsonl", encoding="utf-8") as f:
    print(f.readline().strip())

{"word": "kvinna", "sentence_id": "137d9286_62_69", "label": ["0"]}


In [8]:
import json, zipfile

# 1. Deine bisherige Datei einlesen
with open("subtask1a_predictions.jsonl", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

# 2. Labels in Zahlen umwandeln und unter dem richtigen Namen speichern
with open("SV_subtask1.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        r["label"] = [int(x) for x in r["label"]]
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 3. Ohne Ordner ins ZIP packen
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("SV_subtask1.jsonl", arcname="SV_subtask1.jsonl")

print(zipfile.ZipFile("submission.zip").namelist())   # sollte ['SV_subtask1.jsonl'] zeigen

['SV_subtask1.jsonl']
